# Project Gutenberg Corpus Preparation

This notebook prepares a corpus of English-language short stories from the Project Gutenberg catalog.

## Pipeline Overview
1. Load the Gutenberg metadata index
2. Filter for English-language text entries
3. Explore and filter by genre/bookshelf categories
4. Extract short stories from British and American literature
5. Download full-text files from Project Gutenberg
6. Parse metadata (title, author) from downloaded files
7. Remove Gutenberg boilerplate (headers/footers)
8. Merge text content with metadata into a final dataset


## Step 1: Load and Inspect the Gutenberg Metadata Index

The Gutenberg index CSV (`pg_catalog.csv`) contains one row per text with fields such as `Text#`, `Type`, `Language`, `Title`, `Authors`, `Subjects`, `Bookshelves`, and `LoCC`. Download it from https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv

In [ ]:
import pandas as pd
import csv

# Load the Gutenberg catalog index
# Source: https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv
csv_file_path = "/path/to/pg_catalog.csv"  # <-- update this path

df = pd.read_csv(csv_file_path)

# Preview the first few rows
df.head()


## Step 2: Explore the Raw Catalog

Check how many entries exist, how many languages are represented, and what content types are available.

In [ ]:
# Total number of entries in the catalog
num_rows = df.shape[0]
print(f"Number of rows: {num_rows}")

# Number of distinct languages
num_languages = df["Language"].nunique()
print(f"Number of unique languages: {num_languages}")

# List all content types (Text, Sound, Image, etc.)
unique_types = df["Type"].unique()
print(f"Content types: {unique_types}")


## Step 3: Filter for English-Language Texts

Retain only entries where `Language == 'en'` and `Type == 'Text'` to focus on readable English prose.

In [ ]:
# Keep only English plain-text entries
df_filtered = df[(df["Language"] == "en") & (df["Type"] == "Text")]

# Preview the filtered DataFrame
df_filtered.head()


In [ ]:
# Confirm size and composition of the filtered set
num_rows = df_filtered.shape[0]
print(f"Number of rows: {num_rows}")

num_languages = df_filtered["Language"].nunique()
print(f"Number of unique languages: {num_languages}")

# Inspect the Bookshelves categories present
unique_types = df_filtered["Type"].unique()
print(f"Types: {unique_types}")


## Step 4: Explore Bookshelf Categories

The `Bookshelves` column contains semicolon-separated category tags. Inspect unique values to identify relevant short-story categories.

In [ ]:
# List unique Bookshelves values in the filtered corpus
unique_bookshelves = df_filtered["Bookshelves"].unique()
print(f"Sample Bookshelves values:\n{unique_bookshelves[:20]}")


In [ ]:
# Convert to a Python list and optionally save to a text file for manual review
unique_bookshelves_list = list(df_filtered["Bookshelves"].unique())
print(f"Total unique Bookshelves entries: {len(unique_bookshelves_list)}")

# Save to file for offline inspection (one entry per line)
output_path = "/path/to/unique_bookshelves.txt"  # <-- update this path
with open(output_path, "w", encoding="utf-8") as f:
    for item in unique_bookshelves_list:
        f.write(f"{item}\n")


## Step 5: Filter for Short Stories

Retain only entries whose `Bookshelves` field contains the tag `'Short Stories'`.

In [ ]:
# Keep entries tagged as Short Stories (case-insensitive match)
df_shorts = df_filtered[
    df_filtered["Bookshelves"].str.contains("Short Stories", case=False, na=False)
]

# Preview
df_shorts.head()


In [ ]:
# Inspect the short stories subset
num_rows = df_shorts.shape[0]
print(f"Number of rows: {num_rows}")

num_authors = df_shorts["Authors"].nunique()
print(f"Number of unique authors: {num_authors}")

# Show the unique Bookshelves combinations present in this subset
unique_categories = df_shorts["Bookshelves"].unique()
print(f"Sample category combinations: {unique_categories[:10]}")


In [ ]:
# Save unique Bookshelves combinations for the short stories subset
unique_types_list = list(df_shorts["Bookshelves"].unique())

output_path = "/path/to/unique_bookshelves_filtered_shorts.txt"  # <-- update this path
with open(output_path, "w", encoding="utf-8") as f:
    for item in unique_types_list:
        f.write(f"{item}\n")


## Step 6: Further Filter for British and American Literature

To focus the corpus on a specific literary tradition, retain only short stories tagged with `'British Literature'` or `'American Literature'`.

In [ ]:
# Keep short stories from British or American literature traditions
df_shorts_en = df_shorts[
    df_shorts["Bookshelves"].str.contains(
        "British Literature|American Literature", case=False, na=False
    )
]

# Preview
df_shorts_en.head()


In [ ]:
# Confirm the size of this focused subset
num_rows = df_shorts_en.shape[0]
print(f"Number of rows: {num_rows}")

num_authors = df_shorts_en["Authors"].nunique()
print(f"Number of unique authors: {num_authors}")

unique_categories = df_shorts_en["Bookshelves"].unique()
print(f"Number of unique Bookshelves combinations: {len(unique_categories)}")


In [ ]:
# Save the filtered metadata to CSV for later use
output_csv_path = "/path/to/gutenberg_short_stories_corpus.csv"  # <-- update this path
df_shorts_en.to_csv(output_csv_path, index=False, encoding="utf-8")
print(f"Saved {len(df_shorts_en)} rows to {output_csv_path}")


## Step 7: Download Full-Text Files from Project Gutenberg

For each entry in the filtered corpus, attempt to download the plain-text file from `https://www.gutenberg.org/files/{Text#}/{Text#}-0.txt`.

In [ ]:
import requests
import os
import re

# Directory to store downloaded .txt files
output_folder = "/path/to/downloaded_texts"  # <-- update this path
os.makedirs(output_folder, exist_ok=True)

# Iterate over the filtered corpus
for _, row in df_shorts_en.iterrows():
    text_id = row["Text#"]
    title = row["Title"]

    # Construct the standard Project Gutenberg plain-text URL
    url = f"https://www.gutenberg.org/files/{text_id}/{text_id}-0.txt"

    # Sanitize the title for use as a filename (remove filesystem-unsafe characters)
    safe_title = re.sub(r'[\\/*?:"<>|]', "", title)[:50]
    filename = f"{text_id}_{safe_title}.txt"
    filepath = os.path.join(output_folder, filename)

    # Attempt download; log failures for retry
    try:
        r = requests.get(url)
        r.raise_for_status()
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(r.text)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Failed to download {text_id}: {e}")


## Step 8: Retry Failed Downloads with Alternative URL Patterns

Some texts use different URL suffixes (e.g. `{id}.txt` or `{id}-8.txt`). A secondary pass tries multiple variants for entries that initially failed.

In [ ]:
# Load a CSV of failed downloads (columns: Text#, Title) prepared from the first pass
failed_csv_path = "/path/to/failed_downloads.csv"  # <-- update this path

df_failed = pd.read_csv(failed_csv_path, sep=",", engine="python")
df_failed.head()


In [ ]:
import requests
import os
import re

# Output folder for retried downloads (can be merged with the main folder afterwards)
retry_folder = "/path/to/downloaded_texts_retry"  # <-- update this path
os.makedirs(retry_folder, exist_ok=True)

for _, row in df_failed.iterrows():
    text_id = str(row["Text#"]).strip()
    title = str(row["Title"]).strip()

    safe_title = re.sub(r'[\\/*?:"<>|]', "", title)[:50]
    filename = f"{text_id}_{safe_title}.txt"
    filepath = os.path.join(retry_folder, filename)

    downloaded = False

    # Build a list of URL candidates to try in order:
    # 1) plain <id>.txt
    # 2) <id>-0.txt through <id>-9.txt
    url_variants = [f"https://www.gutenberg.org/files/{text_id}/{text_id}.txt"]
    url_variants += [
        f"https://www.gutenberg.org/files/{text_id}/{text_id}-{suffix}.txt"
        for suffix in range(0, 10)
    ]

    for url in url_variants:
        try:
            r = requests.get(url, timeout=10)
            # Accept response only if status is 200 and content is substantial
            if r.status_code == 200 and len(r.text) > 1000:
                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(r.text)
                print(f"Downloaded {text_id} from {url}")
                downloaded = True
                break
        except requests.RequestException:
            pass  # Try next URL variant

    if not downloaded:
        print(f"Could not download text {text_id}")


## Step 9: Parse Title and Author from Downloaded Files

Project Gutenberg files include a header line in one of several formats. This cell extracts `Title` and `Author` using regex patterns.

In [ ]:
import pandas as pd
import pathlib
import re

# Directory containing all downloaded .txt files (primary + retry batches)
folder_path = "/path/to/downloaded_texts"  # <-- update this path

rows = []

for file_path in pathlib.Path(folder_path).glob("*.txt"):
    # Extract Text# from filename prefix (e.g. '41_The Legend of Sleepy Hollow.txt' -> '41')
    text_id = file_path.stem.split("_")[0]

    # Read the full file content
    content = file_path.read_text(encoding="utf-8", errors="ignore")

    # Inspect only the first 10 lines where Gutenberg headers appear
    header_lines = content.splitlines()[:10]
    header_text = " ".join(header_lines)

    title = None
    author = None

    # Pattern 1: "The Project Gutenberg EBook of TITLE, by AUTHOR"
    m = re.search(
        r"(?:The\s+)?Project Gutenberg(?:'s| eBook,| EBook of)\s+(.*?),\s+by\s+(.*)",
        header_text,
        re.IGNORECASE,
    )
    if m:
        title = m.group(1).strip()
        author = m.group(2).strip()

    # Pattern 2: Multi-line fallback — find title and author from individual lines
    if title is None or author is None:
        for line in header_lines:
            if title is None:
                t_match = re.search(r"Project Gutenberg(?:'s)?\s+(.*)", line, re.IGNORECASE)
                if t_match:
                    title = t_match.group(1).strip(" ,")
            if author is None:
                a_match = re.search(r"\bby\s+(.+)", line, re.IGNORECASE)
                if a_match:
                    author = a_match.group(1).strip()

    # Clean author: remove license boilerplate that sometimes gets captured
    if author:
        author = author.split("\n")[0].strip()
        if re.match(r"^This eBook is for the use of anyone anywhere", author, re.IGNORECASE):
            author = None

    rows.append({
        "Text#": text_id,
        "Title": title,
        "author": author,
        "content": content
    })

# Assemble into a DataFrame
df_texts = pd.DataFrame(rows)
df_texts.head()


## Step 10: Remove Project Gutenberg Headers and Footers

Each downloaded file contains a standard preamble and legal footer. Truncate the content at the footer boundary so only the literary text remains.

In [ ]:
import pathlib

# Directory of downloaded files to clean in-place
folder_path = "/path/to/downloaded_texts"  # <-- update this path

# Known Gutenberg end-of-book markers (check all variants)
footer_variants = [
    "End of Project Gutenberg's",
    "*** END OF THIS PROJECT GUTENBERG EBOOK",
    "End of the Project Gutenberg",
    "End of Project Gutenberg",
    "*** END OF THE PROJECT GUTENBERG EBOOK"
]

for file_path in pathlib.Path(folder_path).glob("*.txt"):
    content = file_path.read_text(encoding="utf-8", errors="ignore")

    # Find the earliest footer marker position
    split_pos = len(content)
    for footer in footer_variants:
        pos = content.find(footer)
        if pos != -1 and pos < split_pos:
            split_pos = pos

    # Truncate and overwrite the file
    if split_pos != len(content):
        content = content[:split_pos].rstrip()
        file_path.write_text(content, encoding="utf-8")
        print(f"Cleaned: {file_path.name}")


## Step 11: Merge Metadata with Text Content

Join the catalog metadata with the parsed title/author information and the full text content into a single DataFrame.

In [ ]:
import pandas as pd

# Load the filtered metadata CSV produced in Step 6
df_corpus = pd.read_csv("/path/to/gutenberg_short_stories_corpus.csv")  # <-- update

# Load the parsed metadata CSV produced in Step 9
df_parsed = pd.read_csv("/path/to/parsed_metadata.csv")  # <-- update

# Merge on the shared Text# identifier (left join keeps all corpus entries)
merged_df = pd.merge(
    df_corpus,
    df_parsed,
    on="Text#",
    how="left"
)

# Save the merged metadata table
merged_df.to_csv("/path/to/gutenberg_merged.csv", index=False)

merged_df.head()


In [ ]:
import os
import re
from collections import defaultdict
import pandas as pd

# ----------------------------
# Configuration
# ----------------------------
corpus_csv   = "/path/to/gutenberg_short_stories_corpus.csv"  # metadata
parsed_csv   = "/path/to/parsed_metadata.csv"                 # parsed title/author
txt_folder   = "/path/to/downloaded_texts"                    # cleaned .txt files
output_csv   = "/path/to/gutenberg_final_corpus.csv"          # output

# ----------------------------
# Load and merge metadata
# ----------------------------
df_corpus = pd.read_csv(corpus_csv)
df_parsed = pd.read_csv(parsed_csv)

df_corpus["Text#"] = df_corpus["Text#"].astype(int)
df_parsed["Text#"] = df_parsed["Text#"].astype(int)

merged_df = pd.merge(df_corpus, df_parsed, on="Text#", how="left")

# ----------------------------
# Read text files and map content to Text#
# ----------------------------
text_content = defaultdict(list)

for filename in os.listdir(txt_folder):
    if not filename.lower().endswith(".txt"):
        continue
    match = re.match(r"^(\d+)", filename)
    if not match:
        continue

    text_id = int(match.group(1))
    file_path = os.path.join(txt_folder, filename)

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text_content[text_id].append(f.read())

# Concatenate multiple files for the same Text# (if any)
text_content = {k: "\n".join(v) for k, v in text_content.items()}

# Add content column
merged_df["content"] = merged_df["Text#"].map(text_content)

# ----------------------------
# Save and report
# ----------------------------
merged_df.to_csv(output_csv, index=False)

print(f"Total rows:           {len(merged_df)}")
print(f"Rows with content:    {merged_df['content'].notna().sum()}")
print(f"Rows missing content: {merged_df['content'].isna().sum()}")
print("\nSample rows with content:")
print(merged_df.loc[merged_df["content"].notna(), ["Text#", "Title"]].head())


## Final Summary

The `merged_df` DataFrame (saved to `gutenberg_final_corpus.csv`) contains:

- **Text#** — Gutenberg text ID
- **Title**, **Authors** — from the catalog index
- **Bookshelves**, **Subjects**, **LoCC** — genre/classification metadata
- **author** (parsed) — extracted from the file header
- **content** — full cleaned literary text

Entries where `content` is `NaN` could not be downloaded; these can be retried manually or excluded from downstream analysis.

In [ ]:
# Quick summary of the final corpus
print(f"Total entries:        {len(merged_df)}")
print(f"Unique authors:       {merged_df['Authors'].nunique()}")
print(f"Entries with text:    {merged_df['content'].notna().sum()}")

merged_df[["Text#", "Title", "Authors", "content"]].dropna(subset=["content"]).head()
